## Bibliotecas

In [1]:
import pandas as pd
import plotly.express as px
from statsmodels.tsa.seasonal import seasonal_decompose
import logging
import datetime
import time

from datetime import datetime

import dash
from dash import dcc, html
from dash.dependencies import Input, Output

import calendar
import pandas as pd
import plotly.express as px
from pyspark.sql import SparkSession
from pyspark.sql.functions import concat, col
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from ipywidgets import interact, widgets
from matplotlib.dates import MonthLocator, DateFormatter
import geopandas as gpd
from shapely.geometry import Point
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import geopandas as gpd
from mpl_toolkits.axes_grid1 import make_axes_locatable
#from fbprophet import Prophet
from sklearn.metrics import mean_squared_error
import numpy as np
import plotly.graph_objects as go
from pyspark.sql import functions as F
from pyspark.sql.functions import year, month
from sklearn.metrics.pairwise import euclidean_distances
from pyspark.sql.functions import year, month, sum as sum_
from scipy.spatial.distance import euclidean
from prophet import Prophet
from pyspark.sql.functions import col, countDistinct , desc , count , length
from pyspark.sql.functions import split, col, when
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, dayofmonth, last_day ,to_date, lit, isnull
from pyspark.sql.types import IntegerType
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error
from dateutil.relativedelta import relativedelta
from pyspark.sql.functions import date_format

from pyspark.sql.functions import col, coalesce, current_date, months_between, round
from pyspark.sql.functions import regexp_replace

import schedule
from pyspark.sql.functions import col, to_timestamp, date_format

import pyspark
import pandas as pd
import boto3
import sagemaker
import sagemaker.feature_store.feature_store as fs
import databricks.connect
from dateutil.relativedelta import relativedelta

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, mean_absolute_error
from sklearn.metrics import mean_squared_error, root_mean_squared_error

import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
import plotly.graph_objects as go
from itertools import product
import random
import statsmodels.api as sm
import os

from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, months_between, current_date,expr

import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
from databricks.connect import DatabricksSession


# Get spark
spark = databricks.connect.DatabricksSession.builder.getOrCreate()

c:\Users\gusta\AppData\Local\pypoetry\Cache\virtualenvs\ltv-novo-aF9Dy8BA-py3.10\lib\site-packages\pydantic\_internal\_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


[02/17/25 11:12:05] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=565221;file://c:\Users\gusta\AppData\Local\pypoetry\Cache\virtualenvs\ltv-novo-aF9Dy8BA-py3.10\lib\site-packages\botocore\credentials.py\credentials.py]8;;\:]8;id=249078;file://c:\Users\gusta\AppData\Local\pypoetry\Cache\virtualenvs\ltv-novo-aF9Dy8BA-py3.10\lib\site-packages\botocore\credentials.py#1278\1278]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\gusta\AppData\Local\sagemaker\sagemaker\config.yaml


In [2]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

### Tabelas Usadas

In [3]:
cliente = spark.table("prd.dl_redeok_checkok.tb_cliente") # Tabela com data inici, data final , motivo do bloqueio e max_conexao = next_status , cnpj
revenda = spark.table("prd.dl_redeok_checkok.tb_revenda") # Só usamos para filtrar os contratos tipo revenda
receber = spark.table("prd.dl_redeok_checkok.vw_receber") # Tabela de faturas
estab_df = spark.table("prd.dl_receita_federal.tb_estabelecimento").alias("estab") \
    .withColumn("CNPJ_CPF", concat(col("cnpj_basico"), col("cnpj_ordem"), col("cnpj_dv")))
hierarquia_sdf = spark.table("dev.dw_rok.tb_cnae_hierarquia_comercial").alias("hierarquia")


In [4]:
# Selecionar as colunas desejadas no estab_df
estab_selected = estab_df.select("CNPJ_CPF", "cnae_fiscal_principal")

# Realizar o join com hierarquia_sdf utilizando a coluna cnae_fiscal_principal e CNAE_RF
resultado_df = estab_selected.join(hierarquia_sdf, estab_selected["cnae_fiscal_principal"] == hierarquia_sdf["CNAE_RF"], "left")

resultado_df = resultado_df.select("CNPJ_CPF", "CNAE_RF", "Hierarquia")


### Querry

In [5]:
query=  """WITH 
-- CTE para obter informações básicas dos clientes, como quantidade de contratos, primeiro e último bloqueio, motivo e status de conexão
clientes AS 
(
    SELECT 
          COUNT(1)              AS contratos          -- Contagem total de contratos por documento
        , cgcmf                 AS documento          -- Documento do cliente (CNPJ)
        , MIN(data_cad)         AS data_inicio        -- Data de início do primeiro contrato do cliente
        , MAX(bloqueio)         AS data_final         -- Data do último bloqueio do cliente
        , MAX_BY(max_conexao, bloqueio) AS max_conexao        -- Status de conexão mais recente
    FROM 
    (
        -- Seleção dos clientes únicos, eliminando duplicatas e aplicando filtros iniciais
        SELECT DISTINCT
            matriz, codigo, fis_jurid, cgcmf, raz_social, max_conexao, bloqueio, mot_bloqueio,
            sit_cliente, data_cad, vendedor, vlrfatmin, vlr_dispacesso, receita
        FROM 
            prd.dl_redeok_checkok.tb_cliente cli
        WHERE 
            cli.sit_cliente IN ('A', 'F', 'N', 'P', 'D')         -- Filtra clientes com status ativos ou pendentes
            AND (cli.matriz = cli.codigo OR falsa = 's')         -- Mantém apenas contas da matriz ou marcadas como falsas
            AND cli.receita IN (9, 10, 46, 58)                   -- Filtra por tipos de receita específico
            AND cli.cgcmf IS NOT NULL                            -- Remove registros sem CNPJ
            AND cli.cgcmf != '23236392000198'                    -- Exclui um CNPJ específico (ROK)
            AND LENGTH(TRIM(cli.cgcmf)) = 14                     -- Garante que o CNPJ tenha 14 caracteres
            AND cli.cgcmf RLIKE '^[0-9]+$'                       -- Garante que o CNPJ seja numérico

            -- Remove registros que fazem parte da tabela de revendas
            AND NOT EXISTS 
            (
                SELECT 1 
                FROM prd.dl_redeok_checkok.tb_revenda rev 
                WHERE cli.codigo = rev.codigo
            )
    ) 
    GROUP BY cgcmf
),

-- CTE para contar a quantidade de boletos gerados por cliente
titulo_cobranca AS 
(
    SELECT 
          documento         AS documento                
        , cliente          AS cliente  
        , COUNT(meses)      AS boletos_gerados        -- Contagem de meses com boletos gerados
    FROM 
    (
    -- Seleção de boletos distintos por cliente e mês
        SELECT DISTINCT
            cli.cgcmf                           AS documento
            , rec.cliente                       AS cliente  
            , DATE_TRUNC('month', rec.emissao)  AS meses     -- Agrupa por mês de emissão do boleto
        FROM 
            prd.dl_redeok_checkok.vw_receber rec
        INNER JOIN prd.dl_redeok_checkok.tb_cliente cli 
            ON cli.codigo = rec.cliente
        WHERE 
            rec.situacao IN ('A', 'B')

             -- Remove registros de revendas
            AND NOT EXISTS 
            (
                SELECT 1 
                FROM prd.dl_redeok_checkok.tb_revenda rev 
                WHERE rev.codigo = rec.cliente
            )

             -- Filtra os mesmos clientes da CTE 'clientes'
            AND cli.sit_cliente IN ('A', 'F', 'N', 'P', 'D')
            AND (cli.matriz = cli.codigo OR falsa = 's')
            AND cli.receita IN (9, 10, 46, 58)
            AND cli.cgcmf IS NOT NULL
            AND cli.cgcmf != '23236392000198'
            AND LENGTH(TRIM(cli.cgcmf)) = 14    
            AND cli.cgcmf RLIKE '^[0-9]+$'
    ) 
    GROUP BY 
        documento, cliente  
),

-- CTE para encontrar a data do último boleto gerado por cliente
ultimo_boleto AS 
(
    SELECT 
        cli.cgcmf AS documento,
        MAX(rec.emissao) AS ultima_emissao            -- Última data de emissão do boleto
    FROM 
        prd.dl_redeok_checkok.vw_receber rec
    INNER JOIN prd.dl_redeok_checkok.tb_cliente cli 
        ON cli.codigo = rec.cliente
    WHERE 
        rec.situacao IN ('A', 'B')                   -- Apenas boletos ativos ou em aberto

        -- Remove registros de revendas
        AND NOT EXISTS 
        (
            SELECT 1 
            FROM prd.dl_redeok_checkok.tb_revenda rev 
            WHERE rev.codigo = rec.cliente
        )
        AND cli.sit_cliente IN ('A', 'F', 'N', 'P', 'D')
        AND (cli.matriz = cli.codigo OR falsa = 's')
        AND cli.receita IN (9, 10, 46, 58)
        AND cli.cgcmf IS NOT NULL
        AND cli.cgcmf != '23236392000198'
        AND LENGTH(TRIM(cli.cgcmf)) = 14    
        AND cli.cgcmf RLIKE '^[0-9]+$'
    GROUP BY cli.cgcmf
),

-- CTE para encontrar a data do primeiro boleto gerado por cliente
primeiro_boleto AS 
(
    SELECT 
        cli.cgcmf AS documento,
        MIN(rec.emissao) AS data_primeiro_boleto
    FROM 
        prd.dl_redeok_checkok.vw_receber rec
    INNER JOIN prd.dl_redeok_checkok.tb_cliente cli 
        ON cli.codigo = rec.cliente
    WHERE 
        rec.situacao IN ('A', 'B')
        AND NOT EXISTS 
        (
            SELECT 1 
            FROM prd.dl_redeok_checkok.tb_revenda rev 
            WHERE rev.codigo = rec.cliente
        )
        AND cli.sit_cliente IN ('A', 'F', 'N', 'P', 'D')
        AND (cli.matriz = cli.codigo OR falsa = 's')
        AND cli.receita IN (9, 10, 46, 58)
        AND cli.cgcmf IS NOT NULL
        AND cli.cgcmf != '23236392000198'
        AND LENGTH(TRIM(cli.cgcmf)) = 14    
        AND cli.cgcmf RLIKE '^[0-9]+$'
    GROUP BY cli.cgcmf
),

-- CTE para somar o total pago por cliente
soma_valor_pago AS
(
    SELECT 
        cli.cgcmf AS documento,
        SUM(rec.valor_pago) AS total_valor_pago       -- Soma total de valores pagos pelo cliente
    FROM 
        prd.dl_redeok_checkok.vw_receber rec
    INNER JOIN prd.dl_redeok_checkok.tb_cliente cli 
        ON cli.codigo = rec.cliente
    WHERE 
        rec.situacao IN ('A', 'B')

        -- Remove registros de revendas
        AND NOT EXISTS 
        (
            SELECT 1 
            FROM prd.dl_redeok_checkok.tb_revenda rev 
            WHERE rev.codigo = rec.cliente
        )
        AND cli.sit_cliente IN ('A', 'F', 'N', 'P', 'D')
        AND (cli.matriz = cli.codigo OR falsa = 's')
        AND cli.receita IN (9, 10, 46, 58)
        AND cli.cgcmf IS NOT NULL
        AND cli.cgcmf != '23236392000198'
        AND LENGTH(TRIM(cli.cgcmf)) = 14    
        AND cli.cgcmf RLIKE '^[0-9]+$'
    GROUP BY cli.cgcmf
),
documentos_filtrados AS (
    SELECT 
          cli.contratos
        , cli.documento
        , tit.cliente  
        , cli.data_inicio
        , cli.data_final  
        , cli.max_conexao 
        , prim.data_primeiro_boleto   
        , CASE 
              WHEN cli.data_final IS NULL THEN NULL
              ELSE ult.ultima_emissao
          END AS data_ultimo_boleto
        , tit.boletos_gerados
        , ROUND(COALESCE(soma.total_valor_pago, 0), 1) AS total_valor_pago
        , ROW_NUMBER() OVER (PARTITION BY cli.documento ORDER BY tit.boletos_gerados DESC) AS row_number
     FROM 
        clientes cli
    INNER JOIN titulo_cobranca tit 
        ON cli.documento = tit.documento
    LEFT JOIN ultimo_boleto ult
        ON cli.documento = ult.documento
    LEFT JOIN primeiro_boleto prim
        ON cli.documento = prim.documento
    LEFT JOIN soma_valor_pago soma
        ON cli.documento = soma.documento
)
SELECT 
    contratos,
    documento,
    cliente,
    data_inicio,
    data_final,
    max_conexao,
    data_primeiro_boleto,
    data_ultimo_boleto,
    boletos_gerados,
    total_valor_pago
FROM documentos_filtrados
WHERE row_number = 1;

"""


# Executando a consulta e obtendo os dados em um DataFrame Spark
tabela_nova_sdf= spark.sql(query)

from pyspark.sql.functions import round

# Criar a coluna 'tipo_saida' com as condições especificadas
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "tipo_saida",
    F.when(F.col("max_conexao") == -1, "CANCELAMENTO NORMAL")
    .when(F.col("max_conexao") == 0, "90 DIAS BLOQUEADO")
    .when(F.col("max_conexao") == -5, "90 DIAS PRÉ CANCELADO")
    .when(F.col("max_conexao") == -7, "90 DIAS PRÉ CANCELADO JURÍDICO")
    .when(F.col("max_conexao").isNull(), "ATIVO")
    .otherwise("OUTROS")
)



from pyspark.sql.functions import col, to_date

from pyspark.sql.functions import col, to_timestamp

# Corrigir datas que possuem o ano como '0012' para '2012' e '0017' para '2017'
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "data_final", 
    regexp_replace("data_final", "^0011-", "2011-")  # Corrige 0012 para 2012
)


# Corrigir datas que possuem o ano como '0012' para '2012' e '0017' para '2017'
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "data_final", 
    regexp_replace("data_final", "^0201-", "2019-")  # Corrige 0012 para 2012
)

# Corrigir datas que possuem o ano como '0012' para '2012' e '0017' para '2017'
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "data_final", 
    regexp_replace("data_final", "^0012-", "2012-")  # Corrige 0012 para 2012
)

tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "data_final", 
    regexp_replace("data_final", "^0017-", "2017-")  # Corrige 0017 para 2017
)

# Adicionar o cálculo do meses_contrato tratando data_final nulo
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "meses_contrato",
    round(
        months_between(
            # Usa a data atual quando data_final for nulo
            coalesce(col("data_final"), current_date()),
            col("data_inicio")
        )
    )
)

# Criar a coluna 'meses_boleto'
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "meses_boleto",
    F.round(
        F.months_between(
            F.coalesce(F.col("data_ultimo_boleto"), F.current_date()),  # Se for nulo, usa a data atual
            F.col("data_primeiro_boleto")
        ), 2  # Arredonda para 2 casas decimais
    )
)

# Criar a coluna 'anos_boleto'
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "anos_boleto",
    F.round(
        F.months_between(
            F.coalesce(F.col("data_ultimo_boleto"), F.current_date()),  # Corrigido para F.col(...)
            F.col("data_primeiro_boleto")  # Corrigido para F.col(...)
        ) / 12, 2  # Divide por 12 para converter para anos e arredonda para 2 casas decimais
    )
)
tabela_nova_sdf = tabela_nova_sdf.withColumn("anos_contrato", round(col("meses_contrato") / 12, 2))

tabela_nova_sdf = tabela_nova_sdf.withColumn("churn",when(col("data_final").isNotNull(), 1).otherwise(0))


tabela_nova_sdf = tabela_nova_sdf.filter(col("contratos") == 1)

# Substituir valores nulos pela data atual
# Criar colunas corrigindo a diferença para ser sempre positiva
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "diferenca_meses_inicio_primeiro_boleto",
    expr("ceil(abs(months_between(data_primeiro_boleto, data_inicio)))")
).withColumn(
    "diferenca_meses_final_ultimo_boleto",
    expr("abs(floor(months_between(COALESCE(data_final, current_date()), COALESCE(data_ultimo_boleto, current_date()))))")
)

from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, expr

# Criar a classificação para diferença de meses
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "classifica_diferenca_boleto_tempo_inicial",
    when(col("diferenca_meses_inicio_primeiro_boleto") <= 1, "0 a 1 mês")
    .when((col("diferenca_meses_inicio_primeiro_boleto") >= 2) & (col("diferenca_meses_inicio_primeiro_boleto") <= 5), "2 a 5 meses")
    .when((col("diferenca_meses_inicio_primeiro_boleto") >= 6) & (col("diferenca_meses_inicio_primeiro_boleto") <= 12), "6 a 12 meses")
    .when((col("diferenca_meses_inicio_primeiro_boleto") > 12) & (col("diferenca_meses_inicio_primeiro_boleto") <= 24), "13 a 24 meses")
    .otherwise("Maior que 24 meses")
).withColumn(
    "classifica_diferenca_boleto_tempo_final",
    when(col("diferenca_meses_final_ultimo_boleto") <= 1, "0 a 1 mês")
    .when((col("diferenca_meses_final_ultimo_boleto") >= 2) & (col("diferenca_meses_final_ultimo_boleto") <= 5), "2 a 5 meses")
    .when((col("diferenca_meses_final_ultimo_boleto") >= 6) & (col("diferenca_meses_final_ultimo_boleto") <= 12), "6 a 12 meses")
    .when((col("diferenca_meses_final_ultimo_boleto") > 12) & (col("diferenca_meses_final_ultimo_boleto") <= 24), "13 a 24 meses")
    .otherwise("Maior que 24 meses")
)

tabela_nova_sdf = tabela_nova_sdf.filter((col("meses_contrato") > 0))

# Pegando o ano das datas
#df = tabela_nova_sdf.withColumn("ano_primeiro_boleto", year(col("data_primeiro_boleto"))) \
       #.withColumn("ano_ultimo_boleto", year(col("data_ultimo_boleto")))

tabela_nova_sdf = tabela_nova_sdf.filter(
    (year(col("data_primeiro_boleto")).isNotNull()) & 
    (year(col("data_primeiro_boleto")) >= 1900) & 
    ((col("data_ultimo_boleto").isNull()) | (year(col("data_ultimo_boleto")) >= 1900)) &  
    (year(col("data_primeiro_boleto")) <= 2100) & 
    ((col("data_ultimo_boleto").isNull()) | (year(col("data_ultimo_boleto")) <= 2100))
)





In [6]:
# Realizar o join entre tabela_nova_sdf e resultado_df usando documento = CNPJ_CPF
tabela_nova_sdf = tabela_nova_sdf.join(resultado_df, tabela_nova_sdf["documento"] == resultado_df["CNPJ_CPF"], "left")


In [7]:
tabela_filtrada_2006 = tabela_nova_sdf.select("contratos", "documento", "cliente","data_inicio","data_final","data_primeiro_boleto","data_ultimo_boleto",
                                         "anos_boleto","anos_contrato","meses_boleto","meses_contrato","max_conexao","tipo_saida", "boletos_gerados","total_valor_pago","churn",
                                         "diferenca_meses_inicio_primeiro_boleto","diferenca_meses_final_ultimo_boleto","classifica_diferenca_boleto_tempo_inicial","classifica_diferenca_boleto_tempo_final","Hierarquia")  


In [8]:
from pyspark.sql.functions import to_timestamp

tabela_nova_sdf = tabela_nova_sdf.withColumn("data_inicio", to_date(tabela_nova_sdf["data_inicio"], "yyyy-MM-dd HH:mm:ss"))
tabela_nova_sdf = tabela_nova_sdf.withColumn("data_final", to_date(tabela_nova_sdf["data_final"], "yyyy-MM-dd HH:mm:ss"))
tabela_nova_sdf = tabela_nova_sdf.withColumn("data_primeiro_boleto", to_date(tabela_nova_sdf["data_primeiro_boleto"], "yyyy-MM-dd HH:mm:ss"))
tabela_nova_sdf = tabela_nova_sdf.withColumn("data_ultimo_boleto", to_date(tabela_nova_sdf["data_ultimo_boleto"], "yyyy-MM-dd HH:mm:ss"))

In [9]:
from pyspark.sql.functions import col, year

# Filtrar as linhas onde pelo menos uma das datas é de 2006 ou posterior
tabela_filtrada_2006 = tabela_nova_sdf.filter(
    (year(col("data_inicio")) >= 2006)
)

tabela_filtrada_2006 = tabela_filtrada_2006.filter(col("documento") != "04192021000107")



In [10]:
tabela_filtrada_2006.show()

+---------+--------------+-------+-----------+----------+-----------+--------------------+------------------+---------------+----------------+-------------------+--------------+------------+-----------+-------------+-----+--------------------------------------+-----------------------------------+-----------------------------------------+---------------------------------------+--------------+-------+--------------------+
|contratos|     documento|cliente|data_inicio|data_final|max_conexao|data_primeiro_boleto|data_ultimo_boleto|boletos_gerados|total_valor_pago|         tipo_saida|meses_contrato|meses_boleto|anos_boleto|anos_contrato|churn|diferenca_meses_inicio_primeiro_boleto|diferenca_meses_final_ultimo_boleto|classifica_diferenca_boleto_tempo_inicial|classifica_diferenca_boleto_tempo_final|      CNPJ_CPF|CNAE_RF|          Hierarquia|
+---------+--------------+-------+-----------+----------+-----------+--------------------+------------------+---------------+----------------+----------

In [11]:
tabela_filtrada_2006.count()

118012

## Analises Exploratorias

In [12]:
# Filtrar as linhas onde 'data_primeiro_boleto' é menor que 2006
tabela_verificacao = tabela_filtrada_2006.filter(year(col("data_primeiro_boleto")) < 2006)


tabela_verificacao.show()
# Contar quantas linhas atendem à condição
quantidade = tabela_verificacao.count()

# Exibir a quantidade de ocorrências
print(f"Existem {quantidade} registros com 'data_primeiro_boleto' menor que 2006.")


+---------+---------+-------+-----------+----------+-----------+--------------------+------------------+---------------+----------------+----------+--------------+------------+-----------+-------------+-----+--------------------------------------+-----------------------------------+-----------------------------------------+---------------------------------------+--------+-------+----------+
|contratos|documento|cliente|data_inicio|data_final|max_conexao|data_primeiro_boleto|data_ultimo_boleto|boletos_gerados|total_valor_pago|tipo_saida|meses_contrato|meses_boleto|anos_boleto|anos_contrato|churn|diferenca_meses_inicio_primeiro_boleto|diferenca_meses_final_ultimo_boleto|classifica_diferenca_boleto_tempo_inicial|classifica_diferenca_boleto_tempo_final|CNPJ_CPF|CNAE_RF|Hierarquia|
+---------+---------+-------+-----------+----------+-----------+--------------------+------------------+---------------+----------------+----------+--------------+------------+-----------+-------------+-----+----

In [13]:
from pyspark.sql.functions import col, sum

# Criar um DataFrame com a contagem de nulos por coluna
nulos_por_coluna = tabela_filtrada_2006.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in tabela_filtrada_2006.columns]
)

# Exibir o resultado
nulos_por_coluna.show()


+---------+---------+-------+-----------+----------+-----------+--------------------+------------------+---------------+----------------+----------+--------------+------------+-----------+-------------+-----+--------------------------------------+-----------------------------------+-----------------------------------------+---------------------------------------+--------+-------+----------+
|contratos|documento|cliente|data_inicio|data_final|max_conexao|data_primeiro_boleto|data_ultimo_boleto|boletos_gerados|total_valor_pago|tipo_saida|meses_contrato|meses_boleto|anos_boleto|anos_contrato|churn|diferenca_meses_inicio_primeiro_boleto|diferenca_meses_final_ultimo_boleto|classifica_diferenca_boleto_tempo_inicial|classifica_diferenca_boleto_tempo_final|CNPJ_CPF|CNAE_RF|Hierarquia|
+---------+---------+-------+-----------+----------+-----------+--------------------+------------------+---------------+----------------+----------+--------------+------------+-----------+-------------+-----+----

### Quantidade por tipo de saida

In [14]:
# Contar a quantidade de ocorrências de cada churn_type e ordenar do maior para o menor
churn_count_df = (
    tabela_filtrada_2006.groupBy("tipo_saida")
    .count()
    .orderBy(F.desc("count"))
)

# Exibir o resultado
churn_count_df.show(truncate=False)

+------------------------------+-----+
|tipo_saida                    |count|
+------------------------------+-----+
|CANCELAMENTO NORMAL           |71708|
|90 DIAS BLOQUEADO             |18393|
|OUTROS                        |13197|
|ATIVO                         |12735|
|90 DIAS PRÉ CANCELADO         |1406 |
|90 DIAS PRÉ CANCELADO JURÍDICO|573  |
+------------------------------+-----+



### Tempo de Permanência por Inicio de contrato: Contratos vs. Boletos 

In [15]:
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime

def analisar_permanencia_comparada(tabela_sdf):
    """
    Analisa e compara o tempo de permanência dos contratos baseados na data de início e na data do primeiro boleto.

    Parâmetros:
    tabela_sdf (Spark DataFrame): DataFrame do Spark contendo 'data_inicio', 'anos_contrato', 
                                  'data_primeiro_boleto' e 'anos_boleto'.

    Retorno:
    fig (plotly.graph_objects.Figure): Gráfico interativo comparando os tempos de permanência.
    """

    # Filtrar e converter para Pandas os contratos cancelados
    df_contratos = tabela_sdf.filter(tabela_sdf["tipo_saida"] != "ATIVO") \
        .select("data_inicio", "anos_contrato", "data_primeiro_boleto", "anos_boleto") \
        .toPandas()

    # Converter colunas de data para datetime
    df_contratos["data_inicio"] = pd.to_datetime(df_contratos["data_inicio"])
    df_contratos["data_primeiro_boleto"] = pd.to_datetime(df_contratos["data_primeiro_boleto"])

    # Criar colunas com o ano do contrato e o ano do primeiro boleto
    df_contratos["ano_inicio"] = df_contratos["data_inicio"].dt.year
    df_contratos["ano_primeiro_boleto"] = df_contratos["data_primeiro_boleto"].dt.year

    # Obter o ano atual
    ano_atual = datetime.today().year

    # Calcular estatísticas por ano para os contratos
    estatisticas_contratos = df_contratos.groupby("ano_inicio")["anos_contrato"].agg(["median", "mean"]).reset_index()
    estatisticas_contratos["tempo_maximo_possivel"] = ano_atual - estatisticas_contratos["ano_inicio"]

    # Calcular estatísticas por ano para os boletos
    estatisticas_boletos = df_contratos.groupby("ano_primeiro_boleto")["anos_boleto"].agg(["median", "mean"]).reset_index()
    estatisticas_boletos["tempo_maximo_possivel"] = ano_atual - estatisticas_boletos["ano_primeiro_boleto"]

    # Criar gráfico interativo
    fig = go.Figure()

    # Adicionar linhas dos contratos
    fig.add_trace(go.Scatter(
        x=estatisticas_contratos["ano_inicio"],
        y=estatisticas_contratos["median"],
        mode="lines+markers",
        name="Mediana Contratos",
        line=dict(color="blue")
    ))

    fig.add_trace(go.Scatter(
        x=estatisticas_contratos["ano_inicio"],
        y=estatisticas_contratos["mean"],
        mode="lines+markers",
        name="Média Contratos",
        line=dict(color="green")
    ))

    fig.add_trace(go.Scatter(
        x=estatisticas_contratos["ano_inicio"],
        y=estatisticas_contratos["tempo_maximo_possivel"],
        mode="lines",
        name="Tempo Máximo Contratos",
        line=dict(color="red", dash="dash")
    ))

    # Adicionar linhas dos boletos
    fig.add_trace(go.Scatter(
        x=estatisticas_boletos["ano_primeiro_boleto"],
        y=estatisticas_boletos["median"],
        mode="lines+markers",
        name="Mediana Boletos",
        line=dict(color="purple")
    ))

    fig.add_trace(go.Scatter(
        x=estatisticas_boletos["ano_primeiro_boleto"],
        y=estatisticas_boletos["mean"],
        mode="lines+markers",
        name="Média Boletos",
        line=dict(color="orange")
    ))

    fig.add_trace(go.Scatter(
        x=estatisticas_boletos["ano_primeiro_boleto"],
        y=estatisticas_boletos["tempo_maximo_possivel"],
        mode="lines",
        name="Tempo Máximo Boletos",
        line=dict(color="brown", dash="dash")
    ))

    # Layout do gráfico
    fig.update_layout(
        title="Comparação do Tempo de Permanência: Contratos vs. Boletos",
        xaxis_title="Ano de Início",
        yaxis_title="Tempo de Permanência (anos)",
        showlegend=True,
        xaxis=dict(
            dtick=1,
            tickmode='linear'
        )
    )

    return fig


analisar_permanencia_comparada(tabela_filtrada_2006)


### Boxplot Tempo de permanencia por Inicio de Contrato

### Quantidde de clientes por Inicio de Contrato

In [ ]:
# Criar DataFrame

df = tabela_nova_sdf

# Converter colunas para tipo data
df = df.withColumn("data_inicio", df["data_inicio"].cast("date"))
df = df.withColumn("data_primeiro_boleto", df["data_primeiro_boleto"].cast("date"))

# Contagem distinta de clientes (documento) por ano da data_inicial
contagem_data_inicial = (
    df.groupBy(year(df["data_inicio"]).alias("ano"))
    .agg(countDistinct("documento").alias("contagem_data_inicial"))
    .orderBy("ano")
)

# Contagem distinta de clientes (documento) por ano da data_primeiro_boleto
contagem_data_primeiro_boleto = (
    df.groupBy(year(df["data_primeiro_boleto"]).alias("ano"))
    .agg(countDistinct("documento").alias("contagem_data_primeiro_boleto"))
    .orderBy("ano")
)

# Exibir resultados
contagem_data_inicial.show(50)
contagem_data_primeiro_boleto.show(50)

### Faixa Tempo de Permanencia

In [ ]:
import plotly.graph_objects as go
import pandas as pd

def plotar_distribuicao_permanencia(sdf, coluna_anos="anos_contrato", coluna_tipo_saida="tipo_saida"):
    """
    Gera um gráfico de barras mostrando a distribuição do tempo de permanência dos clientes cancelados.

    Parâmetros:
    sdf (DataFrame do Spark): DataFrame contendo as colunas de anos de contrato e status de saída.
    coluna_anos (str): Nome da coluna que contém a quantidade de anos do contrato.
    coluna_tipo_saida (str): Nome da coluna que contém o tipo de saída do cliente.

    Retorno:
    None (exibe o gráfico interativo com Plotly)
    """
    
    # Filtrar os clientes cancelados
    sdf_filtrado = sdf.filter(sdf[coluna_tipo_saida] != "ATIVO")

    # Converter para Pandas para visualização
    df_permanencia = sdf_filtrado.select(coluna_anos).toPandas()

    # Definir as faixas de tempo de permanência em anos
    bins = [0, 1, 2, 3, 4, 5, 6, 8, 10, df_permanencia[coluna_anos].max()]
    labels = [
        "0-1 ano",
        "1-2 anos",
        "2-3 anos",
        "3-4 anos",
        "4-5 anos",
        "5-6 anos",
        "6-8 anos",
        "8-10 anos",
        "10+ anos"
    ]

    # Criar coluna categórica com os intervalos
    df_permanencia["faixa_permanencia"] = pd.cut(df_permanencia[coluna_anos], bins=bins, labels=labels, right=True)

    # Contar a frequência de cada faixa
    df_faixas_status = df_permanencia["faixa_permanencia"].value_counts().reset_index()
    df_faixas_status.columns = ["faixa_permanencia", "frequencia"]
    df_faixas_status = df_faixas_status.sort_values(by="faixa_permanencia")  # Ordenar categorias

    # Criar gráfico de barras
    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=df_faixas_status["faixa_permanencia"],
        y=df_faixas_status["frequencia"],
        name="Frequência",
        marker_color="blue"
    ))

    # Configurar layout do gráfico
    fig.update_layout(
        title="Distribuição do Tempo de Permanência dos Clientes Cancelados por Faixa (anos)",
        xaxis_title="Faixa de Tempo de Permanência (anos)",
        yaxis_title="Frequência",
        template="plotly_white"
    )

    # Exibir gráfico
    fig.show()


#### Contratos

In [ ]:
plotar_distribuicao_permanencia(tabela_nova_sdf, coluna_anos="anos_contrato", coluna_tipo_saida="tipo_saida")

#### Boletos

In [ ]:
plotar_distribuicao_permanencia(tabela_nova_sdf, coluna_anos="anos_boleto", coluna_tipo_saida="tipo_saida")

In [ ]:
from pyspark.sql import functions as F

# Contar a quantidade de cada faixa no Spark
count_inicial_sdf = tabela_nova_sdf.groupBy("classifica_diferenca_boleto_tempo_inicial").count()
count_final_sdf = tabela_nova_sdf.groupBy("classifica_diferenca_boleto_tempo_final").count()

# Adicionar a linha do total
total_inicial = count_inicial_sdf.agg(F.sum("count")).collect()[0][0]
total_final = count_final_sdf.agg(F.sum("count")).collect()[0][0]

count_inicial_sdf = count_inicial_sdf.union(
    spark.createDataFrame([("Total", total_inicial)], ["classifica_diferenca_boleto_tempo_inicial", "count"])
)

count_final_sdf = count_final_sdf.union(
    spark.createDataFrame([("Total", total_final)], ["classifica_diferenca_boleto_tempo_final", "count"])
)

# Converter para Pandas
count_inicial_pdf = count_inicial_sdf.toPandas()
count_final_pdf = count_final_sdf.toPandas()




# Ordenar os dados pela quantidade em ordem decrescente (do maior para o menor)
count_inicial_pdf = count_inicial_pdf.sort_values(by="count", ascending=False)
count_final_pdf = count_final_pdf.sort_values(by="count", ascending=False)

# Criar gráfico interativo para classifica_diferenca_boleto_tempo_inicial
fig_inicial = go.Figure()
fig_inicial.add_trace(go.Bar(
    x=count_inicial_pdf["classifica_diferenca_boleto_tempo_inicial"],
    y=count_inicial_pdf["count"],
    marker_color='blue'
))
fig_inicial.update_layout(
    title="Distribuição - Classificação Diferença Boleto Tempo Inicial",
    xaxis_title="Faixa",
    yaxis_title="Quantidade",
    xaxis=dict(categoryorder="total descending")
)
fig_inicial.show()

# Criar gráfico interativo para classifica_diferenca_boleto_tempo_final
fig_final = go.Figure()
fig_final.add_trace(go.Bar(
    x=count_final_pdf["classifica_diferenca_boleto_tempo_final"],
    y=count_final_pdf["count"],
    marker_color='red'
))
fig_final.update_layout(
    title="Distribuição - Classificação Diferença Boleto Tempo Final",
    xaxis_title="Faixa",
    yaxis_title="Quantidade",
    xaxis=dict(categoryorder="total descending")
)
fig_final.show()

display(count_inicial_pdf)
display(count_final_pdf)




## Survivor Analysys